# Intelligent RAG — Zyro Dynamics HR Help Desk
**Goal:** Build a chatbot that answers questions from the 11 supplied HR policy PDFs, cites its sources, and refuses unrelated questions.

> Before running: attach the actual 11 HR policy PDFs. The Loom-guidelines PDF describes the assignment but does not contain the policy dataset.

## 1. Install dependencies

In [1]:
%pip install -q langchain langchain-community langchain-openai langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configure paths and API key

In [4]:
import os
from pathlib import Path

# Kaggle example: set this to the attached dataset folder containing the PDFs.
# Google Colab example: /content/hr_policies
PDF_DIR = Path("data/hr_policies")
INDEX_DIR = Path("faiss_index")
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o-mini"

# Set OPENAI_API_KEY using Kaggle Secrets or Colab Secrets.
# Never paste a real API key into a notebook that you will share.
#assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY using your notebook platform's secret manager."

## 3. Load and inspect the 11 PDF files

In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

pdf_files = list(PDF_DIR.glob("*.pdf"))
print("PDF files found:", len(pdf_files))
for path in pdf_files:
    print("-", path.name)

#assert len(pdf_files) == 11, f"Assignment expects 11 PDFs; found {len(pdf_files)}. Check PDF_DIR."

loader = PyPDFDirectoryLoader(str(PDF_DIR), glob="*.pdf")
documents = loader.load()
print("Pages extracted:", len(documents))
print("Sample metadata:", documents[0].metadata)
print(documents[0].page_content[:1000])

C:\Users\vviki\AppData\Local\Temp\ipykernel_7516\3851280231.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


PDF files found: 11
- 01_Leave_and_Paid_Time_Off_Policy.pdf
- 02_Remote_and_Hybrid_Work_Policy.pdf
- 03_Code_of_Conduct_and_Respectful_Workplace_Policy.pdf
- 04_Compensation_and_Payroll_Policy.pdf
- 05_Travel_and_Expense_Reimbursement_Policy.pdf
- 06_Employee_Benefits_and_Wellbeing_Policy.pdf
- 07_Attendance_and_Working_Hours_Policy.pdf
- 08_Performance_Management_and_Feedback_Policy.pdf
- 09_Learning_and_Professional_Development_Policy.pdf
- 10_Recruitment_and_Employee_Referral_Policy.pdf
- 11_Information_Security_and_Acceptable_Use_Policy.pdf
Pages extracted: 13
Sample metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-14T13:41:38+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-11-14T13:41:38+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data\\hr_policies\\01_Leave_and_Paid_Time_Off_Policy.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
SYNTHETIC TRAI

## 4. Chunk the extracted text

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\\n\\n", "\\n", ". ", " ", ""]
)
chunks = splitter.split_documents(documents)

for chunk in chunks:
    from pathlib import Path
    chunk.metadata["source_file"] = Path(chunk.metadata.get("source", "Unknown")).name
    chunk.metadata["page_number"] = int(chunk.metadata.get("page", 0)) + 1

print("Total chunks:", len(chunks))
print("\n--- Sample chunk ---\n")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Total chunks: 42

--- Sample chunk ---

SYNTHETIC TRAINING DATA — NOT AN OFFICIAL ZYRO DYNAMICS POLICY
Page 1
ZYRO DYNAMICS · HR POLICY LIBRARY
 Leave and Paid Time Off Policy
Policy ID: HR-LEAVE-001     Version: 1.0     Effective date: 14 November 2026
Document status: Synthetic demonstration policy created for an educational RAG project. It is not an official company
policy and must not be used as legal, employment or payroll advice.
1. Purpose and scope
This policy explains planned leave, sick leave, public holidays and leave administration for full-time employees of the
fictional Zyro Dynamics demonstration company. It is synthetic training content, not an actual company policy.
2. Annual leave entitlement
Eligible full-time employees receive 18 working days of paid annual leave per completed leave year

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-14T13:41:38+00:00', 'author': '(anonymous)', 'keywords': '', 'mo

## 5. Create embeddings and the FAISS vector store
The embedding model converts text into vectors. FAISS stores those vectors for semantic retrieval.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(str(INDEX_DIR))
print("FAISS index saved to:", INDEX_DIR)

C:\Users\vviki\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 796.36it/s]


FAISS index saved to: faiss_index


In [8]:
from pathlib import Path

index_path = Path("faiss_index")

print("Index folder exists:", index_path.exists())
print("Index files:", list(index_path.iterdir()))

Index folder exists: True
Index files: [WindowsPath('faiss_index/index.faiss'), WindowsPath('faiss_index/index.pkl')]


## 6. Test MMR retrieval

In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.5}
)

test_question = "What is the company's work from home policy?"
retrieved_docs = retriever.invoke(test_question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\\n--- Result {i} ---")
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print(doc.page_content[:700])

## 7. Add relevance gate, HR guardrail and LCEL answer chain
The score threshold below is a starting point only. Tune it with labeled in-scope and out-of-scope questions.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

RELEVANCE_THRESHOLD = 0.25

def format_context(docs):
    blocks = []
    for i, doc in enumerate(docs, 1):
        blocks.append(
            f"[Source {i}: {doc.metadata.get('source_file')}, "
            f"page {doc.metadata.get('page_number')}]\\n{doc.page_content}"
        )
    return "\\n\\n".join(blocks)

prompt = ChatPromptTemplate.from_template(
    """You are Zyro Dynamics' HR Help Desk assistant.
Answer ONLY questions about the HR policies contained in the context.
Use only the context. Never invent policy details.
If the answer is not present, say the available documents do not provide enough information
and suggest contacting HR. Politely refuse unrelated questions.

Context:
{context}

Question: {question}

Answer:"""
)

llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

answer_chain = (
    {
        "context": RunnableLambda(lambda x: format_context(x["docs"])),
        "question": RunnableLambda(lambda x: x["question"])
    }
    | prompt
    | llm
    | StrOutputParser()
)

def ask_hr(question):
    scored = vector_store.similarity_search_with_relevance_scores(question, k=3)
    best_score = float(scored[0][1]) if scored else 0.0

    if not scored or best_score < RELEVANCE_THRESHOLD:
        return {
            "answer": "I can help with Zyro Dynamics HR policies, but this question does not appear to be covered by the available HR documents. Please ask an HR-related question or contact HR.",
            "sources": [],
            "score": best_score
        }

    docs = retriever.invoke(question)
    answer = answer_chain.invoke({"question": question, "docs": docs})
    sources = [
        (d.metadata.get("source_file"), d.metadata.get("page_number"))
        for d in docs
    ]
    return {"answer": answer, "sources": sources, "score": best_score}

## 8. Test one HR question and one out-of-scope question

In [ ]:
# Replace the first question with one whose answer you have verified in your PDFs.
for q in [
    "What does the Leave Policy say about annual leave?",
    "Who won the last IPL?"
]:
    result = ask_hr(q)
    print("\\nQUESTION:", q)
    print("RELEVANCE SCORE:", result["score"])
    print("ANSWER:", result["answer"])
    print("SOURCES:", result["sources"])

## 9. Evaluate the scope threshold
Create a small validation list from your real policy documents. A good threshold must balance:
- In-scope questions incorrectly refused (false negatives)
- Out-of-scope questions incorrectly answered (false positives)

Do not claim a threshold is effective until you have measured these outcomes.

In [ ]:
validation_questions = [
    # Replace/add questions whose expected label you have checked.
    ("What is the work from home policy?", 1),
    ("Who won the last IPL?", 0),
]

for question, expected_in_scope in validation_questions:
    scored = vector_store.similarity_search_with_relevance_scores(question, k=3)
    score = float(scored[0][1]) if scored else 0.0
    predicted_in_scope = int(score >= RELEVANCE_THRESHOLD)
    print({
        "question": question,
        "expected": expected_in_scope,
        "predicted": predicted_in_scope,
        "top_relevance_score": round(score, 3),
        "correct": predicted_in_scope == expected_in_scope
    })

## 10. Next: launch the Streamlit app
After saving the FAISS index and making `pipeline.py` available in the same project folder:
`streamlit run app.py`

The supplied project ZIP includes `pipeline.py`, `app.py`, and `requirements.txt`.